# EGIS — Free Google Colab T4 GPU Pathology Slide Embedding Extractor (H-optimus-0)

This notebook extracts **deterministic, provenance-tracked pathology image embeddings** from open NCI Imaging Data Commons (**IDC `cptac_luad`** H&E slides) using **Bioptimus `H-optimus-0`** (1.1B ViT-giant, Apache-2.0) on Google Colab's free T4 GPU.

### Security Rule
- **Never** hardcode your Hugging Face token in code or commit it to Git.
- Store `HF_TOKEN` in **Colab Secrets** (left sidebar key icon -> `HF_TOKEN`) after accepting terms at `https://huggingface.co/bioptimus/H-optimus-0`.

### Output
Produces `vision_embeddings_h_optimus_0.parquet` matching EGIS's `VisionEmbeddingAdapter` contract so you can drop it straight into your local EGIS multimodal snapshot.

In [ ]:
!pip install -q torch timm huggingface_hub pyarrow pillow idc-index wsidicom

In [ ]:
import hashlib, json, math, os
from pathlib import Path
import torch, timm
import pyarrow as pa, pyarrow.parquet as pq
from torchvision import transforms
from huggingface_hub import login
from idc_index import IDCClient
from wsidicom import WsiDicom

# Read HF token securely from Colab Secrets or environment
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets before running."
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (T4) in Runtime -> Change runtime type."
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

MODEL_ID = "hf-hub:bioptimus/H-optimus-0"
model = timm.create_model(MODEL_ID, pretrained=True, init_values=1e-5, dynamic_img_size=False).cuda().eval()
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517)),
])
print("Loaded", MODEL_ID, "on", torch.cuda.get_device_name(0))

In [ ]:
# Query open CPTAC-LUAD H&E Slide Microscopy (SM) series from NCI IDC
client = IDCClient()
slides_df = client.sql_query("""
    SELECT PatientID, SeriesInstanceUID, StudyInstanceUID, Modality
    FROM index
    WHERE collection_id = 'cptac_luad' AND Modality = 'SM'
    ORDER BY PatientID, SeriesInstanceUID
""")
print(f"Found {len(slides_df)} CPTAC-LUAD H&E slide series across {slides_df['PatientID'].nunique()} patients")

# Take 1 representative slide per patient (adjust MAX_PATIENTS for full cohort)
MAX_PATIENTS = 10
selected = slides_df.drop_duplicates(subset=["PatientID"]).head(MAX_PATIENTS)

slide_root = Path("./idc_cptac_luad_slides")
slide_root.mkdir(exist_ok=True)
rows = []

for _, srow in selected.iterrows():
    pid = str(srow["PatientID"])
    uid = str(srow["SeriesInstanceUID"])
    out_dir = slide_root / uid
    out_dir.mkdir(exist_ok=True)
    client.download_from_selection(seriesInstanceUID=[uid], downloadDir=str(out_dir))
    with WsiDicom.open(out_dir) as slide:
        thumb = slide.read_thumbnail((224, 224)).convert("RGB")
    img_bytes = thumb.tobytes()
    img_sha = hashlib.sha256(img_bytes).hexdigest()
    tensor = transform(thumb).unsqueeze(0).cuda()
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
        vec = model(tensor)[0].detach().float().cpu().tolist()
    norm = math.sqrt(sum(v * v for v in vec))
    vec_norm = [round(v / norm, 6) for v in vec] if norm > 0 else vec
    rows.append({
        "canonical_patient_id": pid,
        "source_specimen_id": uid,
        "sample_type": "Primary Tumor",
        "slide_id": uid,
        "model_id": MODEL_ID,
        "device": "cuda",
        "image_sha256": img_sha,
        "embedding_dim": len(vec_norm),
        "l2_norm": round(norm, 6),
        "embedding_json": json.dumps(vec_norm),
    })

out_path = Path("vision_embeddings_h_optimus_0.parquet")
pq.write_table(pa.Table.from_pylist(rows), out_path, compression="zstd")
print(f"Wrote {len(rows)} H-optimus-0 embeddings -> {out_path} ({out_path.stat().st_size} bytes)")